In [1]:
# 데이터 처리 및 분석
import pandas as pd
import numpy as np
import warnings

# 경로 설정
import os
import sys

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 모델링
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgbm
from catboost import CatBoostRegressor

# 평가지표
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

# 파이프라인
from sklearn.pipeline import Pipeline

# 출력 설정
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)

# 한글 폰트 설정
import platform

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# 시드 설정
np.random.seed(42)
random_seed = 42

print("-" * 60)
print("라이브러리 로드 완료")
print("한글 폰트 설정 완료!")
print("-" * 60)

------------------------------------------------------------
라이브러리 로드 완료
한글 폰트 설정 완료!
------------------------------------------------------------


In [2]:
if not os.getcwd().endswith("airbnb_price_prediction"):
    os.chdir(os.path.abspath(".."))
print(os.getcwd())

d:\Dev\airbnb_price_prediction


In [ ]:
# 이 경로에서 원하는 데이터로 변경
def load_borough_data(base_path, borough_name):
    """
    특정 자치구 폴더에서 6개의 CSV 파일을 로드하여 반환합니다.
    """

    if borough_name == "manhattan":
        suffix = "mnh"
    elif borough_name == "brooklyn":
        suffix = "bkl"
    else:
        suffix = borough_name

    path = f"{base_path}/{borough_name}"

    X_train = pd.read_csv(f"{path}/X_train_{suffix}.csv")
    y_train = pd.read_csv(f"{path}/y_train_{suffix}.csv").squeeze()
    X_test = pd.read_csv(f"{path}/X_test_{suffix}.csv")
    y_test = pd.read_csv(f"{path}/y_test_{suffix}.csv").squeeze()
    ids_train = pd.read_csv(f"{path}/ids_train_{suffix}.csv")
    ids_test = pd.read_csv(f"{path}/ids_test_{suffix}.csv")

    return X_train, y_train, X_test, y_test, ids_train, ids_test


base = "data"

# 보고 싶은 파트를 바꾸면 됨!
X_train_all, y_train_all, X_test_all, y_test_all, ids_train_all, ids_test_all = load_borough_data(base, "all")
X_train_mnh, y_train_mnh, X_test_mnh, y_test_mnh, ids_train_mnh, ids_test_mnh = load_borough_data(base, "manhattan")
X_train_other, y_train_other, X_test_other, y_test_other, ids_train_other, ids_test_other = load_borough_data(
    base, "other"
)
X_train_bkl, y_train_bkl, X_test_bkl, y_test_bkl, ids_train_bkl, ids_test_bkl = load_borough_data(base, "brooklyn")

In [ ]:
# 1. 모든 모델 로드
xgb_all_model = xgb.XGBRegressor()  # 객체 먼저 생성
xgb_all_model.load_model("results/xgb_model_all.json")
xgb_mnh_model = xgb.XGBRegressor()  # 객체 먼저 생성
xgb_mnh_model.load_model("results/xgb_model_mnh.json")
xgb_bkl_model = xgb.XGBRegressor()  # 객체 먼저 생성
xgb_bkl_model.load_model("results/xgb_model_brooklyn.json")
xgb_other_model = xgb.XGBRegressor()  # 객체 먼저 생성
xgb_other_model.load_model("results/xgb_model_other.json")

# 2. X_test로 predict
y_pred_all = xgb_all_model.predict(X_test_all)
y_pred_mnh = xgb_mnh_model.predict(X_test_mnh)
y_pred_bkl = xgb_bkl_model.predict(X_test_bkl)
y_pred_other = xgb_other_model.predict(X_test_other)

# 3. pd.concat으로 자치구 도시 y 붙이기
y_test_city = np.concatenate([y_pred_mnh, y_pred_bkl, y_pred_other])
y_test_real = np.concatenate([y_test_mnh, y_test_bkl, y_test_other])

# 4. 그러고 평가지표로 한 번 봐보기(y_test, pred_final은 저 y_test_city, real로 바꾸기)
rmse = np.sqrt(mean_squared_error(y_test_city, y_test_real))
rmse2 = np.sqrt(mean_squared_error(np.expm1(y_test_city), np.expm1(y_test_real)))
r2 = r2_score(y_test_city, y_test_real)
mape = mean_absolute_percentage_error(np.expm1(y_test_city), np.expm1(y_test_real))

print(f"Log Root Mean Squared Error: {rmse:.3f}")
print(f"결정 계수 : {r2 * 100:.3f}%")
print()

print(f"원래 가격의 RMSE : {rmse2:.3f}")
print(f"원래 가격의 MAPE : {mape * 100:.3f}%")

Log Root Mean Squared Error: 0.314
결정 계수 : 67.015%

원래 가격의 RMSE : 65.066
원래 가격의 MAPE : 25.365%


In [ ]:
# 각 자치구별로 중요한 변수 탐색도 해보았다.가격이 유의미하게 달랐다면, 성능이 더 좋았을 것이다.
# 공통적으로 주요한 변수들이 많았기 때문에